## 6.2 — Snowflake Adaptive Pricing

Fills `test_price` for **adaptive** Snowflake runs in `{catalog}.{schema}.tco_results`.

### Why
Adaptive (gen2-adaptive) warehouses have **no fixed size**, so the size-based price in
`results_prices` doesn't apply and `6.1` leaves their `test_price` NULL. Instead we read the
**actual credits consumed** from `snowflake.account_usage.query_metering_history`, scoped to
each adaptive warehouse and the time window of each benchmark phase, and convert to USD with the
tier's $/credit.

### Flow
1. Find adaptive run-phases in `results_queries` (`platform='snowflake'`, `warehouse_size_type='gen2_adaptive'`) and their `[min(start_time), max(end_time)]` windows.
2. For each, sum `credits_used` for warehouse `TPCDS_{scale}_{warehouse_size}_{base_slug}` within the window.
3. `test_price = credits × $/credit` (tier price_per_unit from `results_prices`).
4. `UPDATE tco_results` for the matching adaptive rows. Idempotent / re-runnable.

**Prereqs:** run `6` + `6.1` first (so `results_queries` and `tco_results` exist); the SF role needs
`account_usage` read access; query_metering_history is per-query and stays current (~1h latency), so run this once the run has finished.

In [ ]:
%pip install --quiet snowflake-connector-python cryptography
dbutils.library.restartPython()

In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Config + params ---
cfg = load_config()
catalog = cfg["dbx_catalog"]
schema  = cfg["dbx_schema"]
queries_table = f"{catalog}.{schema}.results_queries"
prices_table  = f"{catalog}.{schema}.results_prices"
tco_table     = f"{catalog}.{schema}.tco_results"

dbutils.widgets.text("sf_tier", "Enterprise", "Snowflake tier ($/credit)")
dbutils.widgets.text("tco_region", "us-east-1", "Pricing region")
dbutils.widgets.text("sf_metering_warehouse", "", "Small warehouse to RUN the metering query (required)")
TCO_TIER    = dbutils.widgets.get("sf_tier").strip() or "Enterprise"
TCO_REGION  = dbutils.widgets.get("tco_region").strip() or "us-east-1"
SF_QUERY_WH = dbutils.widgets.get("sf_metering_warehouse").strip()

sf_account  = cfg["sf_account_locator"]
sf_region   = cfg["sf_region"]
sf_user     = cfg["sf_user"]
sf_role     = cfg["sf_role"]
sf_database = cfg["sf_database"]
sf_schema   = cfg["sf_schema"]
print(f"tco_table={tco_table} | tier={TCO_TIER} | region={TCO_REGION} | query_wh={SF_QUERY_WH or '(connection default)'}")

In [ ]:
# --- Snowflake connection (key-pair auth; private key inline from config) ---
import snowflake.connector
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

_key = cfg["sf_auth_key"]
_pem = _key.encode() if isinstance(_key, str) else _key
_pkey = serialization.load_pem_private_key(_pem, password=None, backend=default_backend())
_pkb = _pkey.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
_account = sf_account if "-" in sf_account else f"{sf_account}.{sf_region}"

def sf_connect():
    kw = dict(user=sf_user, account=_account, private_key=_pkb,
              role=sf_role, database=sf_database, schema=sf_schema)
    if SF_QUERY_WH:
        kw["warehouse"] = SF_QUERY_WH
    return snowflake.connector.connect(**kw)

_c = sf_connect()
print("connected as:", _c.cursor().execute("select current_user(), current_role()").fetchone())
_c.close()

In [ ]:
# --- Adaptive run-phases + time windows (UTC epoch) from results_queries ---
runs = spark.sql(f"""
    SELECT scale, warehouse_size, strategy_slug AS arch_ref, CASE WHEN lower(warehouse_size)='adaptive' THEN regexp_replace(strategy_slug,'_ADAPTIVE$','') ELSE strategy_slug END AS wh_slug, data_format, test_type, benchmark_run_id,
           unix_timestamp(min(start_time)) AS win_start_epoch,
           unix_timestamp(max(end_time))   AS win_end_epoch,
           count(*) AS n
    FROM {queries_table}
    WHERE platform = 'snowflake'
      AND warehouse_size_type = 'gen2_adaptive'
      AND scale > 1
      AND label_raw NOT LIKE 'drop_%'
    GROUP BY scale, warehouse_size, strategy_slug, data_format, test_type, benchmark_run_id
""").collect()
print(f"{len(runs)} adaptive run-phase(s) found")
for r in runs:
    print(f"  TPCDS_{r['scale']}_{r['warehouse_size'].upper()}_{r['wh_slug']} [{r['test_type']}] n={r['n']}")

In [ ]:
# --- Pull metered credits per adaptive phase, compute test_price ---
# $/credit is per-credit (size-independent); take it from any snowflake price row for tier/region.
_pp = spark.sql(f"""
    SELECT price_per_unit FROM {prices_table}
    WHERE platform='snowflake' AND region='{TCO_REGION}' AND tier='{TCO_TIER}'
      AND price_per_unit > 0
    LIMIT 1
""").collect()
if not _pp:
    raise ValueError(f"No snowflake price_per_unit for region={TCO_REGION} tier={TCO_TIER} in {prices_table}. Run 6.1 first.")
usd_per_credit = float(_pp[0][0])
print(f"$/credit ({TCO_TIER}, {TCO_REGION}) = {usd_per_credit}")

METERING_SQL = """
    SELECT COALESCE(SUM(credits_used), 0)
    FROM snowflake.account_usage.query_metering_history
    WHERE warehouse_name = %(wh)s
      AND query_start_time >= TO_TIMESTAMP_LTZ(%(s)s)
      AND query_start_time <= TO_TIMESTAMP_LTZ(%(e)s)
"""

conn = sf_connect()
cur = conn.cursor()
updates = []
for r in runs:
    wh = f"TPCDS_{r['scale']}_{r['warehouse_size'].upper()}_{r['wh_slug']}"
    cur.execute(METERING_SQL, {"wh": wh, "s": int(r["win_start_epoch"]), "e": int(r["win_end_epoch"])})
    credits = float(cur.fetchone()[0])
    # TODO: to be rolled back before publication -- retrocompat for legacy adaptive
    # warehouses created under the old size='adaptive' model (named TPCDS_{scale}_ADAPTIVE_{base}).
    if credits == 0:
        _base = r['arch_ref'][:-9] if r['arch_ref'].upper().endswith('_ADAPTIVE') else r['arch_ref']
        _legacy_wh = f"TPCDS_{r['scale']}_ADAPTIVE_{_base}"
        if _legacy_wh != wh:
            cur.execute(METERING_SQL, {"wh": _legacy_wh, "s": int(r["win_start_epoch"]), "e": int(r["win_end_epoch"])})
            _lc = float(cur.fetchone()[0])
            if _lc > 0:
                wh, credits = _legacy_wh, _lc
    price = round(credits * usd_per_credit, 4)
    updates.append((r, wh, credits, price))
    print(f"  {wh} [{r['test_type']}] credits={credits:.4f} -> ${price}")
cur.close(); conn.close()

In [ ]:
# --- Write metered cost into tco_results (idempotent) ---
for r, wh, credits, price in updates:
    rid = r["benchmark_run_id"]
    rid_clause = f"benchmark_run_id <=> '{rid}'" if rid else "benchmark_run_id IS NULL"
    spark.sql(f"""
        UPDATE {tco_table}
        SET test_price = {price}
        WHERE platform = 'snowflake'
          AND warehouse_type = 'gen2_adaptive'
          AND scale = '{r['scale']}'
          AND arch_ref = '{r['arch_ref']}'
          AND data_format = '{r['data_format']}'
          AND test_type = '{r['test_type']}'
          AND {rid_clause}
    """)
print(f"Updated {len(updates)} adaptive tco_results row(s)")
display(spark.sql(f"""
    SELECT platform, data_format, arch_ref, warehouse_type, scale, test_type, duration_seconds, test_price
    FROM {tco_table}
    WHERE warehouse_type = 'gen2_adaptive'
    ORDER BY scale, arch_ref, test_type
"""))